# Tập 1: Chạy mô hình nhận dạng ảnh đầu tiên
[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ngocnguyen88/AI-from-Zero/blob/main/tap01/tap01.ipynb)

Mô hình: **ResNet-18** (torchvision), bộ trọng số **ResNet18_Weights.IMAGENET1K_V1** (huấn luyện sẵn trên ImageNet-1k, 1.000 nhãn).
Chạy được trên CPU, và trên Google Colab miễn phí (đã có sẵn torch + torchvision, không cần cài thêm).
Trên Colab: bấm **Runtime > Run all** (Chạy tất cả), hoặc chạy lần lượt từng ô bằng Shift + Enter. Ô chuẩn bị bên dưới tự tải 4 ảnh thử về thư mục `anh/`. Muốn thử ảnh của bạn: bấm biểu tượng thư mục bên trái, tải ảnh lên rồi sửa tên file ở ô 3.

**Ô chuẩn bị. Lấy ảnh thử** (chạy 1 lần)

In [ ]:
import os, urllib.request                    # tải 4 ảnh thử nếu máy chưa có
os.makedirs('anh', exist_ok=True)
goc = 'https://raw.githubusercontent.com/ngocnguyen88/AI-from-Zero/main/tap01/anh/'
for ten in ['meo.png', 'ca-phe.png', 'ten-lua.png', 'the-mau.png']:
    if os.path.exists('anh/' + ten):
        continue
    for lan in range(3):                         # mạng chập chờn thì thử lại, tối đa 3 lần
        try:
            urllib.request.urlretrieve(goc + ten, 'anh/' + ten + '.tam')
            os.replace('anh/' + ten + '.tam', 'anh/' + ten)
            break
        except Exception as loi:
            print('Tải', ten, 'lỗi, thử lại:', loi)
print(sorted(os.listdir('anh')))

**Ô 1. Nạp thư viện**

In [1]:
import torch                        # tính toán trên các con số
from torchvision import models      # kho mô hình có sẵn
from PIL import Image               # mở ảnh

**Ô 2. Nạp mô hình đã huấn luyện sẵn** (lần đầu sẽ tải trọng số khoảng 45 MB)

In [2]:
w = models.ResNet18_Weights.IMAGENET1K_V1      # bộ trọng số chính thức
model = models.resnet18(weights=w).eval()         # eval = chỉ dùng để đoán, không học thêm

**Ô 3. Nạp một tấm ảnh và tiền xử lý** (đổi tên file thành ảnh của bạn)

In [3]:
img = Image.open('anh/meo.png').convert('RGB')    # đổi tên file ở đây
x = w.transforms()(img).unsqueeze(0)               # co ảnh về 224x224, chuẩn hoá, thêm trục 'số ảnh'
print(x.shape)

torch.Size([1, 3, 224, 224])


**Ô 4. In 5 dự đoán cao nhất kèm phần trăm**

In [4]:
with torch.no_grad():
    p = model(x).softmax(1)[0]                    # 1.000 điểm -> 1.000 xác suất
for s, i in zip(*p.topk(5)):
    print(f'{s.item()*100:5.1f}%  {w.meta["categories"][i]}')

 39.1%  Egyptian cat
 38.6%  tiger cat
 21.8%  tabby
  0.5%  lynx
  0.0%  Persian cat


**Ô thêm. Bên trong mô hình có bao nhiêu con số?**

In [5]:
print(sum(t.numel() for t in model.parameters()), 'tham số')

11689512 tham số
